<a href="https://colab.research.google.com/github/nanasita2002/-/blob/main/%D0%9E%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install gdown --quiet

import re
import numpy as np
import pandas as pd
import gdown
from pathlib import Path
!pip install -q gdown xlsxwriter

In [16]:
#Скачивание файлов
url_prolongations = 'https://drive.google.com/uc?id=1n2CIsPZYOIqMfV7b3aUtp1efc9U96xVt'
url_financial = 'https://drive.google.com/uc?id=1r3grFMz_k-XuWhRNLOAZCoLDdVYg2C7X'

gdown.download(url_prolongations, 'prolongations.csv', quiet=False)
gdown.download(url_financial, 'financial_data.csv', quiet=False)

print("Файлы скачаны.")

Downloading...
From: https://drive.google.com/uc?id=1n2CIsPZYOIqMfV7b3aUtp1efc9U96xVt
To: /content/prolongations.csv
100%|██████████| 35.7k/35.7k [00:00<00:00, 29.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1r3grFMz_k-XuWhRNLOAZCoLDdVYg2C7X
To: /content/financial_data.csv
100%|██████████| 74.3k/74.3k [00:00<00:00, 65.8MB/s]

Файлы скачаны.


In [19]:
#разделяем столбцы по разделителю
def read_csv_auto(path):
    return pd.read_csv(
        path,
        sep=None,
        engine='python',
        dtype={'id': str},
        quotechar='"',
        skipinitialspace=True
    )

prolongations_df = read_csv_auto('prolongations.csv')
financial_df = read_csv_auto('financial_data.csv')

print("prolongations_df:", prolongations_df.shape)
print("financial_df:", financial_df.shape)

display(prolongations_df.head())
display(financial_df.head())

prolongations_df: (477, 3)
financial_df: (451, 19)


,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич
3,87,ноябрь 2022,Соколова Анастасия Викторовна
4,429,ноябрь 2022,Соколова Анастасия Викторовна


,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
3,594,NaN,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
4,665,NaN,"10 000,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [21]:
#Вспомогательные функции
RUS_MONTHS = {
    'январь': 1,
    'февраль': 2,
    'март': 3,
    'апрель': 4,
    'май': 5,
    'июнь': 6,
    'июль': 7,
    'август': 8,
    'сентябрь': 9,
    'октябрь': 10,
    'ноябрь': 11,
    'декабрь': 12
}

RUS_MONTHS_REV = {v: k for k, v in RUS_MONTHS.items()}


def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace('\xa0', ' ').replace('\u202f', ' ')
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def parse_month_text(x):
    """
    'Ноябрь 2022' -> Timestamp('2022-11-01')
    """
    if pd.isna(x):
        return pd.NaT

    s = clean_text(x).lower()
    m = re.match(r'^([а-яё]+)\s+(\d{4})$', s)
    if not m:
        return pd.NaT

    month_name, year = m.groups()
    if month_name not in RUS_MONTHS:
        return pd.NaT

    return pd.Timestamp(int(year), RUS_MONTHS[month_name], 1)


def month_label(ts):
    if pd.isna(ts):
        return ''
    return f"{RUS_MONTHS_REV[ts.month].capitalize()} {ts.year}"


def add_months(ts, n):
    return (pd.Timestamp(ts) + pd.DateOffset(months=n)).normalize()


def parse_amount_cell(x):
    """
    Возвращает:
    - amount: числовая сумма
    - status: amount / zero / blank / stop / unknown
    - is_stop: True/False
    """
    if pd.isna(x):
        return pd.Series({'amount': 0.0, 'status': 'blank', 'is_stop': False})

    s = clean_text(x).lower()

    if s == '':
        return pd.Series({'amount': 0.0, 'status': 'blank', 'is_stop': False})

    if 'стоп' in s or s == 'end' or 'end' in s:
        return pd.Series({'amount': 0.0, 'status': 'stop', 'is_stop': True})

    if s == 'в ноль':
        return pd.Series({'amount': 0.0, 'status': 'zero', 'is_stop': False})

    s_num = s.replace(' ', '').replace(',', '.')

    try:
        val = float(s_num)
        if abs(val) < 1e-12:
            return pd.Series({'amount': 0.0, 'status': 'zero', 'is_stop': False})
        return pd.Series({'amount': val, 'status': 'amount', 'is_stop': False})
    except:
        return pd.Series({'amount': 0.0, 'status': 'unknown', 'is_stop': False})

In [22]:
#Подготовка prolongations
prol = prolongations_df.copy()
prol.columns = [clean_text(c) for c in prol.columns]

# Приводим к единому виду
prol = prol.rename(columns={
    'AM': 'manager',
    'month': 'last_month_text'
})

prol['id'] = prol['id'].astype(str).str.strip()
prol['manager'] = prol['manager'].map(clean_text)
prol['last_month_text'] = prol['last_month_text'].map(clean_text)
prol['last_month'] = prol['last_month_text'].map(parse_month_text)

prol = prol[['id', 'manager', 'last_month_text', 'last_month']].dropna(subset=['id', 'last_month'])
prol = prol.drop_duplicates()

# Проверим дубли id
dup_check = (
    prol.groupby('id')
    .agg(n_manager=('manager', 'nunique'),
         n_last_month=('last_month', 'nunique'))
    .reset_index()
)

problem_ids = dup_check[(dup_check['n_manager'] > 1) | (dup_check['n_last_month'] > 1)]

if len(problem_ids) > 0:
    print("ВНИМАНИЕ: найдены id с разными менеджерами или разными last_month в prolongations.")
    display(problem_ids.head(20))
    # Оставим последнюю запись по last_month, если такие случаи есть
    prol = prol.sort_values(['id', 'last_month']).drop_duplicates('id', keep='last')
else:
    prol = prol.sort_values(['id', 'last_month']).drop_duplicates('id', keep='last')

print("Проектов в prolongations после очистки:", len(prol))
display(prol.head())

ВНИМАНИЕ: найдены id с разными менеджерами или разными last_month в prolongations.


,id,n_manager,n_last_month
6,107,2,4
7,109,1,2
8,112,2,5
9,15,1,5
10,154,3,6
13,180,1,2
14,190,1,2
16,196,2,2
17,199,2,2
19,228,1,2


Проектов в prolongations после очистки: 313


,id,manager,last_month_text,last_month
407,1001,Кузнецов Михаил Иванович,ноябрь 2023,2023-11-01
473,1004,без А/М,декабрь 2023,2023-12-01
447,1006,Смирнова Ольга Владимировна,декабрь 2023,2023-12-01
119,101,Попова Екатерина Николаевна,февраль 2023,2023-02-01
401,1012,Петрова Анна Дмитриевна,ноябрь 2023,2023-11-01


In [23]:
#Подготовка financial_data
fin = financial_df.copy()
fin.columns = [clean_text(c) for c in fin.columns]

# Находим колонки с месяцами
month_cols = [c for c in fin.columns if pd.notna(parse_month_text(c))]
print("Колонки месяцев:", month_cols)

# Приводим id и account
fin['id'] = fin['id'].astype(str).str.strip()
if 'Account' in fin.columns:
    fin['Account'] = fin['Account'].map(clean_text)

# Разворачиваем wide -> long
id_vars = [c for c in ['id', 'Причина дубля', 'Account'] if c in fin.columns]

fin_long = fin.melt(
    id_vars=id_vars,
    value_vars=month_cols,
    var_name='month_text',
    value_name='raw_value'
)

fin_long['month'] = fin_long['month_text'].map(parse_month_text)

parsed = fin_long['raw_value'].apply(parse_amount_cell)
fin_long = pd.concat([fin_long, parsed], axis=1)

fin_long['zero_like'] = fin_long['status'].isin(['zero', 'blank'])
fin_long['positive'] = fin_long['amount'] > 0

display(fin_long.head())

Колонки месяцев: ['Ноябрь 2022', 'Декабрь 2022', 'Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023', 'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023', 'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023', 'Январь 2024', 'Февраль 2024']


,id,Причина дубля,Account,month_text,raw_value,month,amount,status,is_stop,zero_like,positive
0,42,NaN,Васильев Артем Александрович,Ноябрь 2022,"36 220,00",2022-11-01,36220.0,amount,False,False,True
1,657,первая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01,0.0,stop,True,False,False
2,657,вторая часть оплаты,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01,0.0,stop,True,False,False
3,594,NaN,Васильев Артем Александрович,Ноябрь 2022,стоп,2022-11-01,0.0,stop,True,False,False
4,665,NaN,Васильев Артем Александрович,Ноябрь 2022,"10 000,00",2022-11-01,10000.0,amount,False,False,True


In [24]:
#Агрегация отгрузок по id и месяцу
monthly = (
    fin_long
    .groupby(['id', 'month'], as_index=False)
    .agg(
        month_amount=('amount', 'sum'),
        any_stop=('is_stop', 'max'),
        all_zero_like=('zero_like', 'all')
    )
)

display(monthly.head(20))

,id,month,month_amount,any_stop,all_zero_like
0,1001,2022-11-01,0.0,False,True
1,1001,2022-12-01,0.0,False,True
2,1001,2023-01-01,0.0,False,True
3,1001,2023-02-01,0.0,False,True
4,1001,2023-03-01,0.0,False,True
5,1001,2023-04-01,0.0,False,True
6,1001,2023-05-01,0.0,False,True
7,1001,2023-06-01,0.0,False,True
8,1001,2023-07-01,0.0,False,True
9,1001,2023-08-01,0.0,False,True


In [25]:
"""Исключение проектов со стоп/end и расчет базовой отгрузки
Правила:
если у проекта есть стоп / end в последний месяц реализации или раньше — проект исключаем;
базовая отгрузка для коэффициента:
берем сумму в последний месяц реализации;
если в этом месяце по всем частям оплаты 0 (в ноль / пусто / 0), берем предыдущий месяц."""
# Первый месяц, где у проекта встретился stop/end
stop_first = (
    monthly[monthly['any_stop']]
    .groupby('id', as_index=False)['month']
    .min()
    .rename(columns={'month': 'first_stop_month'})
)

projects = prol.merge(stop_first, on='id', how='left')

projects['exclude_stop'] = (
    projects['first_stop_month'].notna() &
    (projects['first_stop_month'] <= projects['last_month'])
)

# Справочники для быстрого доступа
amount_lookup = {(r.id, r.month): r.month_amount for r in monthly.itertuples(index=False)}
zero_lookup = {(r.id, r.month): r.all_zero_like for r in monthly.itertuples(index=False)}

projects['prev_month'] = projects['last_month'].apply(lambda x: add_months(x, -1))

projects['base_last_amount'] = [
    amount_lookup.get((pid, m), 0.0)
    for pid, m in zip(projects['id'], projects['last_month'])
]

projects['base_prev_amount'] = [
    amount_lookup.get((pid, m), 0.0)
    for pid, m in zip(projects['id'], projects['prev_month'])
]

projects['last_month_all_zero'] = [
    zero_lookup.get((pid, m), False)
    for pid, m in zip(projects['id'], projects['last_month'])
]

projects['base_amount'] = np.where(
    (projects['base_last_amount'] == 0) & (projects['last_month_all_zero']),
    projects['base_prev_amount'],
    projects['base_last_amount']
)

projects['base_source'] = np.where(
    (projects['base_last_amount'] == 0) & (projects['last_month_all_zero']),
    'Предыдущий месяц',
    'Последний месяц реализации'
)

projects['valid_base'] = projects['base_amount'] > 0

print("Всего проектов:", len(projects))
print("Исключено из-за stop/end:", projects['exclude_stop'].sum())
print("Без валидной базы:", (~projects['valid_base']).sum())

display(projects.head(20))

Всего проектов: 313
Исключено из-за stop/end: 46
Без валидной базы: 62


,id,manager,last_month_text,last_month,first_stop_month,exclude_stop,prev_month,base_last_amount,base_prev_amount,last_month_all_zero,base_amount,base_source,valid_base
0,1001,Кузнецов Михаил Иванович,ноябрь 2023,2023-11-01,NaT,False,2023-10-01,290000.00,0.00,False,290000.00,Последний месяц реализации,True
1,1004,без А/М,декабрь 2023,2023-12-01,NaT,False,2023-11-01,0.00,34000.00,True,34000.00,Предыдущий месяц,True
2,1006,Смирнова Ольга Владимировна,декабрь 2023,2023-12-01,NaT,False,2023-11-01,140495.00,140495.00,False,140495.00,Последний месяц реализации,True
3,101,Попова Екатерина Николаевна,февраль 2023,2023-02-01,2023-02-01,True,2023-01-01,0.00,43800.00,False,0.00,Последний месяц реализации,False
4,1012,Петрова Анна Дмитриевна,ноябрь 2023,2023-11-01,NaT,False,2023-10-01,98492.00,0.00,False,98492.00,Последний месяц реализации,True
5,1015,Смирнова Ольга Владимировна,ноябрь 2023,2023-11-01,NaT,False,2023-10-01,163700.00,0.00,False,163700.00,Последний месяц реализации,True
6,107,Кузнецов Михаил Иванович,ноябрь 2023,2023-11-01,NaT,False,2023-10-01,59885.85,58485.96,False,59885.85,Последний месяц реализации,True
7,109,Васильев Артем Александрович,декабрь 2023,2023-12-01,NaT,False,2023-11-01,94000.00,94000.00,False,94000.00,Последний месяц реализации,True
8,112,Михайлов Андрей Сергеевич,декабрь 2023,2023-12-01,NaT,False,2023-11-01,132410.00,132410.00,False,132410.00,Последний месяц реализации,True
9,15,Иванова Мария Сергеевна,июнь 2023,2023-06-01,NaT,False,2023-05-01,102433.75,138158.00,False,102433.75,Последний месяц реализации,True


In [26]:
#Формирование событий пролонгации M+1 и M+2
ANALYSIS_YEAR = 2023

rows = []

for r in projects.itertuples(index=False):
    if r.exclude_stop:
        continue
    if not r.valid_base:
        continue

    m1 = add_months(r.last_month, 1)
    m2 = add_months(r.last_month, 2)

    amt_m1 = amount_lookup.get((r.id, m1), 0.0)
    amt_m2 = amount_lookup.get((r.id, m2), 0.0)

    # 1-й месяц пролонгации
    if m1.year == ANALYSIS_YEAR:
        rows.append({
            'id': r.id,
            'manager': r.manager,
            'last_month': r.last_month,
            'analysis_month': m1,
            'coef_type': '1-й месяц',
            'base_amount': r.base_amount,
            'renew_amount': amt_m1 if amt_m1 > 0 else 0.0,
            'renewed_flag': int(amt_m1 > 0)
        })

    # 2-й месяц: только если не было пролонгации в 1-й месяц
    if (amt_m1 <= 0) and (m2.year == ANALYSIS_YEAR):
        rows.append({
            'id': r.id,
            'manager': r.manager,
            'last_month': r.last_month,
            'analysis_month': m2,
            'coef_type': '2-й месяц',
            'base_amount': r.base_amount,
            'renew_amount': amt_m2 if amt_m2 > 0 else 0.0,
            'renewed_flag': int(amt_m2 > 0)
        })

events = pd.DataFrame(rows)

events['month_label'] = events['analysis_month'].map(month_label)
events['last_month_label'] = events['last_month'].map(month_label)

print("Событий для расчета:", len(events))
display(events.head(20))

Событий для расчета: 310


,id,manager,last_month,analysis_month,coef_type,base_amount,renew_amount,renewed_flag,month_label,last_month_label
0,1001,Кузнецов Михаил Иванович,2023-11-01,2023-12-01,1-й месяц,290000.00,0.00,0,Декабрь 2023,Ноябрь 2023
1,1012,Петрова Анна Дмитриевна,2023-11-01,2023-12-01,1-й месяц,98492.00,109442.52,1,Декабрь 2023,Ноябрь 2023
2,1015,Смирнова Ольга Владимировна,2023-11-01,2023-12-01,1-й месяц,163700.00,15000.00,1,Декабрь 2023,Ноябрь 2023
3,107,Кузнецов Михаил Иванович,2023-11-01,2023-12-01,1-й месяц,59885.85,65222.34,1,Декабрь 2023,Ноябрь 2023
4,15,Иванова Мария Сергеевна,2023-06-01,2023-07-01,1-й месяц,102433.75,0.00,0,Июль 2023,Июнь 2023
5,15,Иванова Мария Сергеевна,2023-06-01,2023-08-01,2-й месяц,102433.75,0.00,0,Август 2023,Июнь 2023
6,174,Иванова Мария Сергеевна,2023-02-01,2023-03-01,1-й месяц,127200.00,0.00,0,Март 2023,Февраль 2023
7,174,Иванова Мария Сергеевна,2023-02-01,2023-04-01,2-й месяц,127200.00,0.00,0,Апрель 2023,Февраль 2023
8,193,Иванова Мария Сергеевна,2022-12-01,2023-01-01,1-й месяц,190280.00,0.00,0,Январь 2023,Декабрь 2022
9,193,Иванова Мария Сергеевна,2022-12-01,2023-02-01,2-й месяц,190280.00,0.00,0,Февраль 2023,Декабрь 2022


In [27]:
#Агрегация: менеджеры и отдел
def make_long_report(events_df):
    agg = (
        events_df
        .groupby(['manager', 'analysis_month', 'coef_type'], as_index=False)
        .agg(
            projects=('id', 'nunique'),
            renewed_projects=('renewed_flag', 'sum'),
            denominator=('base_amount', 'sum'),
            numerator=('renew_amount', 'sum')
        )
    )

    agg['count_rate'] = np.where(
        agg['projects'] > 0,
        agg['renewed_projects'] / agg['projects'],
        np.nan
    )

    agg['coefficient'] = np.where(
        agg['denominator'] > 0,
        agg['numerator'] / agg['denominator'],
        np.nan
    )

    return agg.sort_values(['manager', 'analysis_month', 'coef_type'])


manager_monthly_long = make_long_report(events)

dept_events = events.copy()
dept_events['manager'] = 'ВСЕГО ПО ОТДЕЛУ'
dept_monthly_long = make_long_report(dept_events)

display(manager_monthly_long.head(20))
display(dept_monthly_long.head(20))

,manager,analysis_month,coef_type,projects,renewed_projects,denominator,numerator,count_rate,coefficient
0,Васильев Артем Александрович,2023-01-01,1-й месяц,4,0,217965.0,0.0,0.000000,0.000000
1,Васильев Артем Александрович,2023-01-01,2-й месяц,4,0,260862.0,0.0,0.000000,0.000000
2,Васильев Артем Александрович,2023-02-01,1-й месяц,1,0,60900.0,0.0,0.000000,0.000000
3,Васильев Артем Александрович,2023-02-01,2-й месяц,4,1,217965.0,44775.0,0.250000,0.205423
4,Васильев Артем Александрович,2023-03-01,1-й месяц,3,0,324580.0,0.0,0.000000,0.000000
5,Васильев Артем Александрович,2023-03-01,2-й месяц,1,0,60900.0,0.0,0.000000,0.000000
6,Васильев Артем Александрович,2023-04-01,1-й месяц,5,0,333583.0,0.0,0.000000,0.000000
7,Васильев Артем Александрович,2023-04-01,2-й месяц,3,0,324580.0,0.0,0.000000,0.000000
8,Васильев Артем Александрович,2023-05-01,1-й месяц,2,0,599850.0,0.0,0.000000,0.000000
9,Васильев Артем Александрович,2023-05-01,2-й месяц,5,0,333583.0,0.0,0.000000,0.000000


,manager,analysis_month,coef_type,projects,renewed_projects,denominator,numerator,count_rate,coefficient
0,ВСЕГО ПО ОТДЕЛУ,2023-01-01,1-й месяц,26,2,2553253.47,327471.67,0.076923,0.128257
1,ВСЕГО ПО ОТДЕЛУ,2023-01-01,2-й месяц,7,0,422732.00,0.00,0.000000,0.000000
2,ВСЕГО ПО ОТДЕЛУ,2023-02-01,1-й месяц,9,2,1300045.50,811155.00,0.222222,0.623944
3,ВСЕГО ПО ОТДЕЛУ,2023-02-01,2-й месяц,24,2,2168272.76,91770.00,0.083333,0.042324
4,ВСЕГО ПО ОТДЕЛУ,2023-03-01,1-й месяц,8,0,806955.00,0.00,0.000000,0.000000
5,ВСЕГО ПО ОТДЕЛУ,2023-03-01,2-й месяц,7,1,464960.50,55225.00,0.142857,0.118774
6,ВСЕГО ПО ОТДЕЛУ,2023-04-01,1-й месяц,13,0,1125438.00,0.00,0.000000,0.000000
7,ВСЕГО ПО ОТДЕЛУ,2023-04-01,2-й месяц,8,0,806955.00,0.00,0.000000,0.000000
8,ВСЕГО ПО ОТДЕЛУ,2023-05-01,1-й месяц,4,0,936710.00,0.00,0.000000,0.000000
9,ВСЕГО ПО ОТДЕЛУ,2023-05-01,2-й месяц,13,0,1125438.00,0.00,0.000000,0.000000


In [28]:
#Удобный формат помесячного отчета
all_2023_months = pd.date_range('2023-01-01', '2023-12-01', freq='MS')
all_managers = sorted(projects['manager'].dropna().unique().tolist())


def reshape_monthly(long_df, manager_list):
    m1 = long_df[long_df['coef_type'] == '1-й месяц'].copy()
    m2 = long_df[long_df['coef_type'] == '2-й месяц'].copy()

    m1 = m1.rename(columns={
        'projects': 'projects_m1',
        'renewed_projects': 'renewed_projects_m1',
        'denominator': 'denominator_m1',
        'numerator': 'numerator_m1',
        'count_rate': 'count_rate_m1',
        'coefficient': 'coefficient_m1'
    })

    m2 = m2.rename(columns={
        'projects': 'projects_m2',
        'renewed_projects': 'renewed_projects_m2',
        'denominator': 'denominator_m2',
        'numerator': 'numerator_m2',
        'count_rate': 'count_rate_m2',
        'coefficient': 'coefficient_m2'
    })

    keep1 = ['manager', 'analysis_month', 'projects_m1', 'renewed_projects_m1',
             'denominator_m1', 'numerator_m1', 'count_rate_m1', 'coefficient_m1']
    keep2 = ['manager', 'analysis_month', 'projects_m2', 'renewed_projects_m2',
             'denominator_m2', 'numerator_m2', 'count_rate_m2', 'coefficient_m2']

    out = pd.merge(
        m1[keep1],
        m2[keep2],
        on=['manager', 'analysis_month'],
        how='outer'
    )

    # Полная сетка менеджер x месяцы 2023
    grid = pd.MultiIndex.from_product(
        [manager_list, all_2023_months],
        names=['manager', 'analysis_month']
    ).to_frame(index=False)

    out = grid.merge(out, on=['manager', 'analysis_month'], how='left')

    count_money_cols = [
        'projects_m1', 'renewed_projects_m1', 'denominator_m1', 'numerator_m1',
        'projects_m2', 'renewed_projects_m2', 'denominator_m2', 'numerator_m2'
    ]

    for c in count_money_cols:
        out[c] = out[c].fillna(0)

    out['month'] = out['analysis_month'].map(month_label)

    out = out[[
        'manager', 'month',
        'projects_m1', 'renewed_projects_m1', 'denominator_m1', 'numerator_m1', 'count_rate_m1', 'coefficient_m1',
        'projects_m2', 'renewed_projects_m2', 'denominator_m2', 'numerator_m2', 'count_rate_m2', 'coefficient_m2'
    ]].sort_values(['manager', 'month'])

    return out


manager_monthly = reshape_monthly(manager_monthly_long, all_managers)
dept_monthly = reshape_monthly(dept_monthly_long, ['ВСЕГО ПО ОТДЕЛУ'])

display(manager_monthly.head(20))
display(dept_monthly.head(12))
"""Годовой коэффициент считаем как отношение суммы числителей к сумме знаменателей за год,
а не как среднее арифметическое месячных коэффициентов. Это корректнее.

,manager,month,projects_m1,renewed_projects_m1,denominator_m1,numerator_m1,count_rate_m1,coefficient_m1,projects_m2,renewed_projects_m2,denominator_m2,numerator_m2,count_rate_m2,coefficient_m2
7,Васильев Артем Александрович,Август 2023,3.0,1.0,454500.00,135000.00,0.333333,0.297030,4.0,0.0,492017.60,0.0,0.00,0.000000
3,Васильев Артем Александрович,Апрель 2023,5.0,0.0,333583.00,0.00,0.000000,0.000000,3.0,0.0,324580.00,0.0,0.00,0.000000
11,Васильев Артем Александрович,Декабрь 2023,3.0,1.0,169720.00,55000.00,0.333333,0.324063,2.0,1.0,180547.50,87540.0,0.50,0.484859
6,Васильев Артем Александрович,Июль 2023,4.0,0.0,492017.60,0.00,0.000000,0.000000,2.0,0.0,230686.00,0.0,0.00,0.000000
5,Васильев Артем Александрович,Июнь 2023,2.0,0.0,230686.00,0.00,0.000000,0.000000,2.0,0.0,599850.00,0.0,0.00,0.000000
4,Васильев Артем Александрович,Май 2023,2.0,0.0,599850.00,0.00,0.000000,0.000000,5.0,0.0,333583.00,0.0,0.00,0.000000
2,Васильев Артем Александрович,Март 2023,3.0,0.0,324580.00,0.00,0.000000,0.000000,1.0,0.0,60900.00,0.0,0.00,0.000000
10,Васильев Артем Александрович,Ноябрь 2023,2.0,0.0,180547.50,0.00,0.000000,0.000000,5.0,0.0,210185.00,0.0,0.00,0.000000
9,Васильев Артем Александрович,Октябрь 2023,5.0,0.0,210185.00,0.00,0.000000,0.000000,3.0,0.0,987965.00,0.0,0.00,0.000000
8,Васильев Артем Александрович,Сентябрь 2023,3.0,0.0,987965.00,0.00,0.000000,0.000000,2.0,0.0,274500.00,0.0,0.00,0.000000


,manager,month,projects_m1,renewed_projects_m1,denominator_m1,numerator_m1,count_rate_m1,coefficient_m1,projects_m2,renewed_projects_m2,denominator_m2,numerator_m2,count_rate_m2,coefficient_m2
7,ВСЕГО ПО ОТДЕЛУ,Август 2023,12,4,1124555.83,282433.88,0.333333,0.251151,13,0,1443814.85,0.0,0.000000,0.000000
3,ВСЕГО ПО ОТДЕЛУ,Апрель 2023,13,0,1125438.00,0.00,0.000000,0.000000,8,0,806955.00,0.0,0.000000,0.000000
11,ВСЕГО ПО ОТДЕЛУ,Декабрь 2023,26,13,2420564.91,1222684.86,0.500000,0.505124,5,1,411862.50,87540.0,0.200000,0.212547
6,ВСЕГО ПО ОТДЕЛУ,Июль 2023,15,2,1594519.85,136320.00,0.133333,0.085493,16,1,1020325.53,69385.0,0.062500,0.068003
5,ВСЕГО ПО ОТДЕЛУ,Июнь 2023,16,0,1020325.53,0.00,0.000000,0.000000,4,0,936710.00,0.0,0.000000,0.000000
4,ВСЕГО ПО ОТДЕЛУ,Май 2023,4,0,936710.00,0.00,0.000000,0.000000,13,0,1125438.00,0.0,0.000000,0.000000
2,ВСЕГО ПО ОТДЕЛУ,Март 2023,8,0,806955.00,0.00,0.000000,0.000000,7,1,464960.50,55225.0,0.142857,0.118774
10,ВСЕГО ПО ОТДЕЛУ,Ноябрь 2023,7,2,563088.97,157070.00,0.285714,0.278943,17,1,1130392.68,21400.0,0.058824,0.018931
9,ВСЕГО ПО ОТДЕЛУ,Октябрь 2023,25,8,2324356.93,1415939.95,0.320000,0.609175,12,0,1907829.88,0.0,0.000000,0.000000
8,ВСЕГО ПО ОТДЕЛУ,Сентябрь 2023,15,3,2189189.88,314920.00,0.200000,0.143852,8,0,684329.63,0.0,0.000000,0.000000


In [29]:
def make_yearly_long(events_df):
    agg = (
        events_df
        .groupby(['manager', 'coef_type'], as_index=False)
        .agg(
            projects=('id', 'nunique'),
            renewed_projects=('renewed_flag', 'sum'),
            denominator=('base_amount', 'sum'),
            numerator=('renew_amount', 'sum')
        )
    )

    agg['count_rate'] = np.where(
        agg['projects'] > 0,
        agg['renewed_projects'] / agg['projects'],
        np.nan
    )

    agg['coefficient'] = np.where(
        agg['denominator'] > 0,
        agg['numerator'] / agg['denominator'],
        np.nan
    )

    return agg


def reshape_yearly(yearly_long_df, manager_list):
    m1 = yearly_long_df[yearly_long_df['coef_type'] == '1-й месяц'].copy()
    m2 = yearly_long_df[yearly_long_df['coef_type'] == '2-й месяц'].copy()

    m1 = m1.rename(columns={
        'projects': 'projects_m1',
        'renewed_projects': 'renewed_projects_m1',
        'denominator': 'denominator_m1',
        'numerator': 'numerator_m1',
        'count_rate': 'count_rate_m1',
        'coefficient': 'coefficient_m1'
    })

    m2 = m2.rename(columns={
        'projects': 'projects_m2',
        'renewed_projects': 'renewed_projects_m2',
        'denominator': 'denominator_m2',
        'numerator': 'numerator_m2',
        'count_rate': 'count_rate_m2',
        'coefficient': 'coefficient_m2'
    })

    base = pd.DataFrame({'manager': manager_list})

    out = (
        base
        .merge(m1[['manager', 'projects_m1', 'renewed_projects_m1', 'denominator_m1', 'numerator_m1', 'count_rate_m1', 'coefficient_m1']],
               on='manager', how='left')
        .merge(m2[['manager', 'projects_m2', 'renewed_projects_m2', 'denominator_m2', 'numerator_m2', 'count_rate_m2', 'coefficient_m2']],
               on='manager', how='left')
    )

    count_money_cols = [
        'projects_m1', 'renewed_projects_m1', 'denominator_m1', 'numerator_m1',
        'projects_m2', 'renewed_projects_m2', 'denominator_m2', 'numerator_m2'
    ]

    for c in count_money_cols:
        out[c] = out[c].fillna(0)

    out.insert(1, 'year', 2023)

    return out


manager_yearly_long = make_yearly_long(events)
dept_yearly_long = make_yearly_long(dept_events)

manager_yearly = reshape_yearly(manager_yearly_long, all_managers)
dept_yearly = reshape_yearly(dept_yearly_long, ['ВСЕГО ПО ОТДЕЛУ'])

display(manager_yearly)
display(dept_yearly)

,manager,year,projects_m1,renewed_projects_m1,denominator_m1,numerator_m1,count_rate_m1,coefficient_m1,projects_m2,renewed_projects_m2,denominator_m2,numerator_m2,count_rate_m2,coefficient_m2
0,Васильев Артем Александрович,2023,37.0,2.0,4262499.10,190000.00,0.054054,0.044575,37.0,2.0,4173641.10,132315.0,0.054054,0.031703
1,Иванова Мария Сергеевна,2023,29.0,4.0,3029806.66,411900.55,0.137931,0.135949,25.0,0.0,2451624.75,0.0,0.000000,0.000000
2,Кузнецов Михаил Иванович,2023,6.0,3.0,575333.38,221702.34,0.500000,0.385346,1.0,0.0,1425.00,0.0,0.000000,0.000000
3,Михайлов Андрей Сергеевич,2023,9.0,2.0,1818799.68,768820.00,0.222222,0.422707,7.0,0.0,1056004.68,0.0,0.000000,0.000000
4,Петрова Анна Дмитриевна,2023,1.0,1.0,98492.00,109442.52,1.000000,1.111182,0.0,0.0,0.00,0.0,NaN,NaN
5,Попова Екатерина Николаевна,2023,33.0,8.0,2171440.26,649246.20,0.242424,0.298993,25.0,1.0,1568685.26,21400.0,0.040000,0.013642
6,Смирнова Ольга Владимировна,2023,24.0,7.0,2093031.05,1225383.75,0.291667,0.585459,11.0,2.0,672205.80,124610.0,0.181818,0.185375
7,Соколова Анастасия Викторовна,2023,37.0,9.0,3909601.74,1091500.00,0.243243,0.279184,28.0,1.0,2600036.74,46995.0,0.035714,0.018075
8,Федорова Марина Васильевна,2023,0.0,0.0,0.00,0.00,NaN,NaN,0.0,0.0,0.00,0.0,NaN,NaN
9,без А/М,2023,0.0,0.0,0.00,0.00,NaN,NaN,0.0,0.0,0.00,0.0,NaN,NaN


,manager,year,projects_m1,renewed_projects_m1,denominator_m1,numerator_m1,count_rate_m1,coefficient_m1,projects_m2,renewed_projects_m2,denominator_m2,numerator_m2,count_rate_m2,coefficient_m2
0,ВСЕГО ПО ОТДЕЛУ,2023,176,36,17959003.87,4667995.36,0.204545,0.259925,134,6,12523623.33,325320.0,0.044776,0.025977


In [31]:
#выгрузка в эксель
output_file = 'prolongation_report_2023.xlsx'

readme = pd.DataFrame({
    'Параметр': [
        'Год анализа',
        'Источник менеджера',
        'Коэффициент 1-го месяца',
        'Коэффициент 2-го месяца',
        'Исключение stop/end',
        'Правило "в ноль"',
        'Годовой коэффициент',
        'Лист контроля'
    ],
    'Описание': [
        '2023',
        'Берется из prolongations.csv (поле AM), т.к. оно первично',
        'Сумма отгрузки проектов, пролонгированных в 1-й месяц / сумма отгрузки последнего месяца реализации завершившихся проектов',
        'Сумма отгрузки проектов, пролонгированных во 2-й месяц / сумма отгрузки последнего месяца реализации проектов, не пролонгированных в 1-й месяц',
        'Если у проекта есть stop/end в последний месяц реализации или раньше — проект исключается из расчета',
        'Если в последний месяц реализации по всем частям оплаты 0, для базы берется предыдущий месяц',
        'Считается как sum(числитель) / sum(знаменатель), а не среднее месячных коэффициентов',
        'Контрольный лист с детализацией по каждому проекту'
    ]
})

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    readme.to_excel(writer, sheet_name='README', index=False)
    manager_monthly.to_excel(writer, sheet_name='Менеджеры_помесячно', index=False)
    dept_monthly.to_excel(writer, sheet_name='Отдел_помесячно', index=False)
    manager_yearly.to_excel(writer, sheet_name='Менеджеры_год', index=False)
    dept_yearly.to_excel(writer, sheet_name='Отдел_год', index=False)
    projects_check.to_excel(writer, sheet_name='Контроль_проектов', index=False)

    workbook = writer.book

    money_fmt = workbook.add_format({'num_format': '#,##0.00'})
    pct_fmt = workbook.add_format({'num_format': '0.0%'})
    header_fmt = workbook.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'fg_color': '#D9EAF7',
        'border': 1
    })

    sheets = {
        'README': readme,
        'Менеджеры_помесячно': manager_monthly,
        'Отдел_помесячно': dept_monthly,
        'Менеджеры_год': manager_yearly,
        'Отдел_год': dept_yearly,
        'Контроль_проектов': projects_check
    }

    percent_cols = {
        'count_rate_m1', 'coefficient_m1',
        'count_rate_m2', 'coefficient_m2'
    }

    money_cols = {
        'base_last_amount', 'base_prev_amount', 'base_amount',
        'm1_amount', 'm2_amount',
        'denominator_m1', 'numerator_m1',
        'denominator_m2', 'numerator_m2'
    }

    for sheet_name, df in sheets.items():
        ws = writer.sheets[sheet_name]

        # Формат заголовков
        for col_num, value in enumerate(df.columns.values):
            ws.write(0, col_num, value, header_fmt)

        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, max(len(df), 1), len(df.columns) - 1)

        for i, col in enumerate(df.columns):
            if len(df) > 0:
                max_len = max(
                    len(str(col)),
                    min(40, int(df[col].astype(str).str.len().quantile(0.95)) + 2)
                )
            else:
                max_len = len(str(col)) + 2

            fmt = None
            if col in percent_cols:
                fmt = pct_fmt
                width = max(12, max_len)
            elif col in money_cols:
                fmt = money_fmt
                width = max(14, max_len)
            else:
                width = max(12, max_len)

            ws.set_column(i, i, width, fmt)

print(f"Отчет сохранен: {output_file}")

Отчет сохранен: prolongation_report_2023.xlsx


In [32]:
from google.colab import files

# Автоматически скачивает файл на ваш компьютер
files.download(output_file)
print(f"✅ {output_file} скачивается...")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ prolongation_report_2023.xlsx скачивается...


In [35]:
# -*- coding: utf-8 -*-
"""
Создание аналитических графиков для отчета по пролонгациям
Отдельный файл с визуализацией
"""

import pandas as pd
import numpy as np
import xlsxwriter
from google.colab import files

# ============================================
# 1. ЗАГРУЗКА ДАННЫХ ИЗ ОСНОВНОГО ОТЧЕТА
# ============================================

print("Загрузка данных из основного отчета...")

# Загружаем ранее созданный отчет
try:
    # Если отчет уже существует, загружаем его
    manager_monthly = pd.read_excel('prolongation_report_2023.xlsx', sheet_name='Менеджеры_помесячно')
    dept_monthly = pd.read_excel('prolongation_report_2023.xlsx', sheet_name='Отдел_помесячно')
    manager_yearly = pd.read_excel('prolongation_report_2023.xlsx', sheet_name='Менеджеры_год')
    dept_yearly = pd.read_excel('prolongation_report_2023.xlsx', sheet_name='Отдел_год')
    print("✅ Данные загружены успешно")
except:
    print("⚠️ Файл отчета не найден. Использую данные из памяти...")
    # Если файла нет, используем данные из переменных (если они есть в памяти)

# ============================================
# 2. ПОДГОТОВКА ДАННЫХ ДЛЯ ГРАФИКОВ
# ============================================

print("\nПодготовка данных для графиков...")

# Очищаем данные от NaN и бесконечностей
def clean_for_chart(df, columns):
    """Заменяет NaN и Inf на 0 в указанных колонках"""
    df_clean = df.copy()
    for col in columns:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna(0)
            df_clean[col] = df_clean[col].replace([np.inf, -np.inf], 0)
    return df_clean

# Очищаем данные по отделам
dept_clean = clean_for_chart(dept_monthly, ['coefficient_m1', 'coefficient_m2'])

# Очищаем данные по менеджерам
managers_clean = clean_for_chart(manager_yearly, ['coefficient_m1', 'coefficient_m2'])

# Получаем данные по отделам для динамики
dept_trend = dept_clean[dept_clean['manager'] == 'ВСЕГО ПО ОТДЕЛУ'].copy()
dept_trend = dept_trend[['month', 'coefficient_m1', 'coefficient_m2']].sort_values('month')

# Получаем топ-10 менеджеров по 1-му коэффициенту
top10_m1 = managers_clean[
    managers_clean['manager'] != 'ВСЕГО ПО ОТДЕЛУ'
].nlargest(10, 'coefficient_m1')[['manager', 'coefficient_m1']].copy()

# Получаем топ-10 менеджеров по 2-му коэффициенту
top10_m2 = managers_clean[
    managers_clean['manager'] != 'ВСЕГО ПО ОТДЕЛУ'
].nlargest(10, 'coefficient_m2')[['manager', 'coefficient_m2']].copy()

# Данные для сравнения коэффициентов (топ-15 по сумме базы)
managers_for_compare = managers_clean[
    managers_clean['manager'] != 'ВСЕГО ПО ОТДЕЛУ'
].copy()
managers_for_compare = managers_for_compare.nlargest(15, 'denominator_m1')[['manager', 'coefficient_m1', 'coefficient_m2']]

print(f"✅ Данные готовы: {len(dept_trend)} месяцев, {len(top10_m1)} менеджеров в топе")

# ============================================
# 3. СОЗДАНИЕ EXCEL ФАЙЛА С ГРАФИКАМИ
# ============================================

output_file = 'prolongation_charts.xlsx'
print(f"\nСоздание файла с графиками: {output_file}")

with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:

    # Сохраняем исходные данные для справки
    dept_trend.to_excel(writer, sheet_name='Данные_динамика', index=False)
    top10_m1.to_excel(writer, sheet_name='Данные_топ10_m1', index=False)
    top10_m2.to_excel(writer, sheet_name='Данные_топ10_m2', index=False)
    managers_for_compare.to_excel(writer, sheet_name='Данные_сравнение', index=False)

    # Создаем лист для графиков
    charts_sheet = writer.book.add_worksheet('Графики')
    workbook = writer.book

    # Форматы для заголовков
    title_fmt = workbook.add_format({
        'bold': True,
        'font_size': 14,
        'font_color': '#1F4E79',
        'align': 'center',
        'valign': 'vcenter'
    })

    subtitle_fmt = workbook.add_format({
        'bold': True,
        'font_size': 11,
        'font_color': '#2E75B6',
        'align': 'left'
    })

    # ============================================
    # ГРАФИК 1: Динамика коэффициентов отдела
    # ============================================

    charts_sheet.write('A1', '📈 Аналитический отчет по пролонгациям за 2023 год', title_fmt)
    charts_sheet.write('A3', '1. Динамика коэффициентов пролонгации отдела', subtitle_fmt)

    # Записываем данные для графика
    charts_sheet.write('A5', 'Месяц')
    charts_sheet.write('B5', 'Пролонгация 1-й месяц')
    charts_sheet.write('C5', 'Пролонгация 2-й месяц')

    months = dept_trend['month'].tolist()
    coeff_m1 = dept_trend['coefficient_m1'].tolist()
    coeff_m2 = dept_trend['coefficient_m2'].tolist()

    for i, (m, m1, m2) in enumerate(zip(months, coeff_m1, coeff_m2)):
        charts_sheet.write(i+6, 0, m)
        charts_sheet.write(i+6, 1, m1)
        charts_sheet.write(i+6, 2, m2)

    # Создаем линейный график
    chart1 = workbook.add_chart({'type': 'line'})

    chart1.add_series({
        'name': 'Пролонгация в 1-й месяц',
        'categories': '=Графики!$A$6:$A$17',
        'values': '=Графики!$B$6:$B$17',
        'line': {'color': '#2E86AB', 'width': 2.5},
        'marker': {'type': 'circle', 'size': 7, 'fill': {'color': '#2E86AB'}},
        'data_labels': {'value': True, 'num_format': '0.0%', 'position': 'above'}
    })

    chart1.add_series({
        'name': 'Пролонгация во 2-й месяц',
        'categories': '=Графики!$A$6:$A$17',
        'values': '=Графики!$C$6:$C$17',
        'line': {'color': '#A23B72', 'width': 2.5, 'dash_type': 'dash'},
        'marker': {'type': 'square', 'size': 7, 'fill': {'color': '#A23B72'}},
        'data_labels': {'value': True, 'num_format': '0.0%', 'position': 'above'}
    })

    chart1.set_title({
        'name': 'Динамика коэффициентов пролонгации отдела за 2023 год',
        'name_font': {'size': 12, 'bold': True}
    })
    chart1.set_x_axis({'name': 'Месяц', 'name_font': {'size': 10}})
    chart1.set_y_axis({
        'name': 'Коэффициент пролонгации (%)',
        'name_font': {'size': 10},
        'num_format': '0.0%',
        'min': 0,
        'max': 1
    })
    chart1.set_legend({'position': 'bottom'})
    chart1.set_size({'width': 800, 'height': 450})

    charts_sheet.insert_chart('E5', chart1)

    # ============================================
    # ГРАФИК 2: Топ-10 менеджеров (1-й месяц)
    # ============================================

    charts_sheet.write('A22', '2. Топ-10 менеджеров по пролонгации в 1-й месяц', subtitle_fmt)

    # Записываем данные
    charts_sheet.write('A24', 'Менеджер')
    charts_sheet.write('B24', 'Коэффициент пролонгации')

    for i, row in enumerate(top10_m1.itertuples()):
        charts_sheet.write(i+25, 0, row.manager)
        charts_sheet.write(i+25, 1, row.coefficient_m1)

    # Создаем горизонтальную гистограмму
    chart2 = workbook.add_chart({'type': 'bar'})

    chart2.add_series({
        'name': 'Годовой коэффициент пролонгации (1-й месяц)',
        'categories': '=Графики!$A$25:$A$34',
        'values': '=Графики!$B$25:$B$34',
        'fill': {'color': '#2E86AB'},
        'data_labels': {'value': True, 'num_format': '0.0%', 'position': 'outside_end'}
    })

    chart2.set_title({
        'name': 'Топ-10 менеджеров по пролонгации в 1-й месяц',
        'name_font': {'size': 12, 'bold': True}
    })
    chart2.set_x_axis({
        'name': 'Коэффициент пролонгации (%)',
        'num_format': '0.0%'
    })
    chart2.set_y_axis({'name': 'Аккаунт-менеджер'})
    chart2.set_size({'width': 700, 'height': 400})

    charts_sheet.insert_chart('E22', chart2)

    # ============================================
    # ГРАФИК 3: Топ-10 менеджеров (2-й месяц)
    # ============================================

    charts_sheet.write('A44', '3. Топ-10 менеджеров по пролонгации во 2-й месяц', subtitle_fmt)

    # Записываем данные
    charts_sheet.write('A46', 'Менеджер')
    charts_sheet.write('B46', 'Коэффициент пролонгации')

    for i, row in enumerate(top10_m2.itertuples()):
        charts_sheet.write(i+47, 0, row.manager)
        charts_sheet.write(i+47, 1, row.coefficient_m2)

    # Создаем горизонтальную гистограмму
    chart3 = workbook.add_chart({'type': 'bar'})

    chart3.add_series({
        'name': 'Годовой коэффициент пролонгации (2-й месяц)',
        'categories': '=Графики!$A$47:$A$56',
        'values': '=Графики!$B$47:$B$56',
        'fill': {'color': '#A23B72'},
        'data_labels': {'value': True, 'num_format': '0.0%', 'position': 'outside_end'}
    })

    chart3.set_title({
        'name': 'Топ-10 менеджеров по пролонгации во 2-й месяц',
        'name_font': {'size': 12, 'bold': True}
    })
    chart3.set_x_axis({
        'name': 'Коэффициент пролонгации (%)',
        'num_format': '0.0%'
    })
    chart3.set_y_axis({'name': 'Аккаунт-менеджер'})
    chart3.set_size({'width': 700, 'height': 400})

    charts_sheet.insert_chart('E44', chart3)

    # ============================================
    # ГРАФИК 4: Сравнение коэффициентов
    # ============================================

    charts_sheet.write('A66', '4. Сравнение коэффициентов пролонгации по менеджерам', subtitle_fmt)

    # Записываем данные
    charts_sheet.write('A68', 'Менеджер')
    charts_sheet.write('B68', 'Пролонгация 1-й месяц')
    charts_sheet.write('C68', 'Пролонгация 2-й месяц')

    for i, row in enumerate(managers_for_compare.itertuples()):
        charts_sheet.write(i+69, 0, row.manager)
        charts_sheet.write(i+69, 1, row.coefficient_m1)
        charts_sheet.write(i+69, 2, row.coefficient_m2)

    # Создаем группированную гистограмму
    chart4 = workbook.add_chart({'type': 'column'})

    chart4.add_series({
        'name': 'Пролонгация в 1-й месяц',
        'categories': '=Графики!$A$69:$A$83',
        'values': '=Графики!$B$69:$B$83',
        'fill': {'color': '#2E86AB'},
        'data_labels': {'value': True, 'num_format': '0.0%', 'position': 'outside_end'}
    })

    chart4.add_series({
        'name': 'Пролонгация во 2-й месяц',
        'categories': '=Графики!$A$69:$A$83',
        'values': '=Графики!$C$69:$C$83',
        'fill': {'color': '#A23B72'},
        'data_labels': {'value': True, 'num_format': '0.0%', 'position': 'outside_end'}
    })

    chart4.set_title({
        'name': 'Сравнение коэффициентов пролонгации (топ-15 по объему базы)',
        'name_font': {'size': 12, 'bold': True}
    })
    chart4.set_x_axis({
        'name': 'Аккаунт-менеджер',
        'label_rotation': 45,
        'label_font': {'size': 9}
    })
    chart4.set_y_axis({
        'name': 'Коэффициент пролонгации (%)',
        'num_format': '0.0%'
    })
    chart4.set_legend({'position': 'top'})
    chart4.set_size({'width': 1000, 'height': 500})

    charts_sheet.insert_chart('E66', chart4)

    # ============================================
    # ГРАФИК 5: Круговая диаграмма - структура пролонгаций
    # ============================================

    charts_sheet.write('A93', '5. Структура пролонгаций по отделу за 2023 год', subtitle_fmt)

    # Получаем годовые итоги отдела
    dept_yearly_clean = clean_for_chart(dept_yearly, ['prolonged_first_month_sum', 'prolonged_second_month_sum'])

    if len(dept_yearly_clean) > 0:
        prolonged_m1 = dept_yearly_clean.iloc[0]['prolonged_first_month_sum'] if 'prolonged_first_month_sum' in dept_yearly_clean.columns else 0
        prolonged_m2 = dept_yearly_clean.iloc[0]['prolonged_second_month_sum'] if 'prolonged_second_month_sum' in dept_yearly_clean.columns else 0
        not_prolonged = dept_yearly_clean.iloc[0]['denominator_m1'] - prolonged_m1 if 'denominator_m1' in dept_yearly_clean.columns else 0

        # Записываем данные для круговой диаграммы
        charts_sheet.write('A95', 'Категория')
        charts_sheet.write('B95', 'Сумма')
        charts_sheet.write('A96', 'Пролонгировано в 1-й месяц')
        charts_sheet.write('B96', prolonged_m1)
        charts_sheet.write('A97', 'Пролонгировано во 2-й месяц')
        charts_sheet.write('B97', prolonged_m2)
        charts_sheet.write('A98', 'Не пролонгировано')
        charts_sheet.write('B98', not_prolonged)



    # Настройка ширины колонок
    charts_sheet.set_column('A:A', 30)
    charts_sheet.set_column('B:C', 18)
    charts_sheet.set_column('E:E', 30)

print(f"\n✅ Файл с графиками сохранен: {output_file}")

# ============================================
# 4. АВТОМАТИЧЕСКОЕ СКАЧИВАНИЕ
# ============================================

print("\nСкачивание файла...")
files.download(output_file)
print("✅ Готово! Файл с графиками загружен на ваш компьютер")

Загрузка данных из основного отчета...
✅ Данные загружены успешно

Подготовка данных для графиков...
✅ Данные готовы: 12 месяцев, 10 менеджеров в топе

Создание файла с графиками: prolongation_charts.xlsx

✅ Файл с графиками сохранен: prolongation_charts.xlsx

Скачивание файла...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Готово! Файл с графиками загружен на ваш компьютер
